# 19｜不用 `nn.RNN/LSTM/GRU`：从门公式实现循环网络

这不是“调一个循环层 API”的演示，而是一份可以逐行审计的最小工程实现。我们会手写 `CustomRNNCell`、`CustomLSTMCell`、`CustomGRUCell` 和变长序列扫描器，并检查参数量、张量形状、mask、梯度、受控过拟合、状态化推理与模型指纹。

> 边界：实验使用 CPU 与固定的小型合成数据，证明的是实现能够学习且合同成立，不代表真实语料上的泛化效果。

## 1. 验收目标与 API 合同

- 输入统一为 `x: [B,T,D]`，长度为 `lengths: [B]`，且每条长度必须位于 `[1,T]`。
- cell 输入一个时间步 `x_t: [B,D]`；RNN/GRU 状态是 `[B,H]`，LSTM 状态是 `(h,c)`。
- 扫描器在 padding 位置**不得更新状态**，输出 padding 位置归零。
- 分类器只消费最后一个有效状态，避免把填充值当作特征。
- 所有随机源固定；训练仅做“固定小集合受控过拟合”。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import hashlib
import io
import json
import math
import random
from dataclasses import dataclass

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260728
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert torch.__version__
assert DEVICE.type == "cpu"
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. Vanilla RNN：最短的循环记忆

对时间步 $t$：

$$h_t=\tanh(W_xx_t+W_hh_{t-1}+b).$$

若输入维度为 $D$、隐藏维度为 $H$，这里把输入偏置放在 `x2h`，因此参数量是 $H(D+H+1)$。`forward` 只描述单步；时间展开稍后由扫描器显式完成。

In [ ]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        if input_size <= 0 or hidden_size <= 0:
            raise ValueError("input_size 和 hidden_size 必须为正")
        self.input_size, self.hidden_size = input_size, hidden_size
        self.x2h = nn.Linear(input_size, hidden_size, bias=True)
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        if x_t.ndim != 2 or h_prev.ndim != 2:
            raise ValueError("单步输入和状态都必须是二维张量")
        if x_t.shape[0] != h_prev.shape[0]:
            raise ValueError("batch 维不一致")
        return torch.tanh(self.x2h(x_t) + self.h2h(h_prev))

rnn_cell = CustomRNNCell(3, 5)
sample_h = rnn_cell(torch.zeros(2, 3), torch.zeros(2, 5))
assert sample_h.shape == (2, 5)
assert torch.isfinite(sample_h).all()
assert sum(p.numel() for p in rnn_cell.parameters()) == 5 * (3 + 5 + 1)

## 3. LSTM：输入、遗忘、候选与输出门

令 $u_t=[x_t;h_{t-1}]$，一次仿射变换得到四组值：

$$i_t=\sigma(W_i u_t+b_i),\quad f_t=\sigma(W_f u_t+b_f),$$
$$g_t=\tanh(W_g u_t+b_g),\quad o_t=\sigma(W_o u_t+b_o),$$
$$c_t=f_t\odot c_{t-1}+i_t\odot g_t,\quad h_t=o_t\odot\tanh(c_t).$$

拼接实现只是计算优化，不改变公式；参数量为 $4H(D+H+1)$。门顺序在类中固定为 `i,f,g,o`，这是 checkpoint 合同的一部分。

In [ ]:
class CustomLSTMCell(nn.Module):
    gate_order = ("input", "forget", "candidate", "output")

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        if input_size <= 0 or hidden_size <= 0:
            raise ValueError("维度必须为正")
        self.input_size, self.hidden_size = input_size, hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x_t, state):
        h_prev, c_prev = state
        if x_t.ndim != 2 or h_prev.shape != c_prev.shape:
            raise ValueError("LSTM 状态必须是同形状的二维 (h,c)")
        packed = self.gates(torch.cat([x_t, h_prev], dim=-1))
        i_raw, f_raw, g_raw, o_raw = packed.chunk(4, dim=-1)
        i, f, o = torch.sigmoid(i_raw), torch.sigmoid(f_raw), torch.sigmoid(o_raw)
        g = torch.tanh(g_raw)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

lstm_cell = CustomLSTMCell(3, 5)
h1, c1 = lstm_cell(torch.randn(2, 3), (torch.zeros(2, 5), torch.zeros(2, 5)))
assert h1.shape == c1.shape == (2, 5)
assert CustomLSTMCell.gate_order == ("input", "forget", "candidate", "output")
assert sum(p.numel() for p in lstm_cell.parameters()) == 4 * 5 * (3 + 5 + 1)

## 4. GRU：两扇门合并状态

本实现采用常见的 reset-after 形式：

$$z_t=\sigma(W_zx_t+U_zh_{t-1}),\quad r_t=\sigma(W_rx_t+U_rh_{t-1}),$$
$$n_t=\tanh(W_nx_t+r_t\odot U_nh_{t-1}),$$
$$h_t=(1-z_t)\odot n_t+z_t\odot h_{t-1}.$$

不同框架可能把 reset 放在矩阵乘法之前，迁移权重时必须记录变体。本实现的参数量为 $3H(D+H+1)$。

In [ ]:
class CustomGRUCell(nn.Module):
    variant = "reset_after"

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size, self.hidden_size = input_size, hidden_size
        self.x_proj = nn.Linear(input_size, 3 * hidden_size, bias=True)
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        if x_t.ndim != 2 or h_prev.ndim != 2:
            raise ValueError("GRU 单步张量必须二维")
        x_z, x_r, x_n = self.x_proj(x_t).chunk(3, dim=-1)
        h_z, h_r, h_n = self.h_proj(h_prev).chunk(3, dim=-1)
        z = torch.sigmoid(x_z + h_z)
        r = torch.sigmoid(x_r + h_r)
        n = torch.tanh(x_n + r * h_n)
        return (1.0 - z) * n + z * h_prev

gru_cell = CustomGRUCell(3, 5)
gru_h = gru_cell(torch.randn(2, 3), torch.zeros(2, 5))
assert gru_h.shape == (2, 5)
assert CustomGRUCell.variant == "reset_after"
assert sum(p.numel() for p in gru_cell.parameters()) == 3 * 5 * (3 + 5 + 1)

## 5. 显式时间扫描与变长 mask

`lengths > t` 生成 `[B,1]` 活跃掩码。新状态只写入活跃样本，已结束样本保留最后有效状态；逐步输出则把 padding 位置清零。这两个动作不能混为一谈：前者保证最终表示正确，后者避免下游误读 padding。

循环在 Python 层实现，便于教学和审计；生产中长序列应替换为融合 kernel 或经过验证的官方实现。

In [ ]:
CELL_TYPES = {"rnn": CustomRNNCell, "lstm": CustomLSTMCell, "gru": CustomGRUCell}

class SequenceEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, kind="lstm"):
        super().__init__()
        if kind not in CELL_TYPES:
            raise ValueError(f"未知 cell: {kind}")
        self.kind, self.hidden_size = kind, hidden_size
        self.cell = CELL_TYPES[kind](input_size, hidden_size)

    def initial_state(self, batch, *, device, dtype):
        z = torch.zeros(batch, self.hidden_size, device=device, dtype=dtype)
        return (z, z.clone()) if self.kind == "lstm" else z

    def forward(self, x, lengths, state=None):
        if x.ndim != 3 or lengths.shape != (x.shape[0],):
            raise ValueError("期望 x=[B,T,D], lengths=[B]")
        B, T, _ = x.shape
        if bool(((lengths < 1) | (lengths > T)).any()):
            raise ValueError("lengths 必须位于 [1,T]")
        state = self.initial_state(B, device=x.device, dtype=x.dtype) if state is None else state
        outputs = []
        for t in range(T):
            active = (lengths > t).unsqueeze(-1)
            if self.kind == "lstm":
                old_h, old_c = state
                new_h, new_c = self.cell(x[:, t], state)
                state = (torch.where(active, new_h, old_h), torch.where(active, new_c, old_c))
                out_t = torch.where(active, state[0], torch.zeros_like(state[0]))
            else:
                new_h = self.cell(x[:, t], state)
                state = torch.where(active, new_h, state)
                out_t = torch.where(active, state, torch.zeros_like(state))
            outputs.append(out_t)
        return torch.stack(outputs, dim=1), state

class SequenceClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, kind="lstm"):
        super().__init__()
        self.encoder = SequenceEncoder(input_size, hidden_size, kind)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        sequence, state = self.encoder(x, lengths)
        final_h = state[0] if self.encoder.kind == "lstm" else state
        return self.head(final_h), sequence

for kind in CELL_TYPES:
    enc = SequenceEncoder(3, 7, kind)
    seq, state = enc(torch.randn(4, 6, 3), torch.tensor([6, 4, 2, 1]))
    assert seq.shape == (4, 6, 7)
    assert torch.count_nonzero(seq[3, 1:]) == 0

## 6. 可诊断的合成任务

每个样本第一维的有效时间步平均值决定二分类标签；padding 故意填入大噪声。模型若正确使用长度就只能从有效片段学习。我们只在固定的 40 条样本上训练并验收，明确称为**受控过拟合**，不报告“测试集泛化”。

In [ ]:
def make_toy_batch(n=40, max_len=7, dim=3):
    g = torch.Generator().manual_seed(SEED + 1)
    lengths = torch.randint(2, max_len + 1, (n,), generator=g)
    x = torch.randn(n, max_len, dim, generator=g)
    valid = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)
    signal = (x[:, :, 0] * valid).sum(1) / lengths
    y = (signal > 0).long()
    # padding 的第一维加巨大干扰；正确 mask 后不应影响标签。
    pad = ~valid
    x[:, :, 0] = torch.where(pad, 25.0 * torch.randn(n, max_len, generator=g), x[:, :, 0])
    return x, lengths, y, valid

x_train, len_train, y_train, valid_train = make_toy_batch()
assert x_train.shape == (40, 7, 3)
assert len_train.min().item() >= 2
assert set(y_train.tolist()) == {0, 1}
assert (~valid_train).any()
print({"samples": len(y_train), "positive_rate": y_train.float().mean().item()})

## 7. 优化器、损失与梯度合同

使用交叉熵和 Adam。每步在 `backward` 后先检查梯度有限且非零，再用全局范数裁剪限制爆炸梯度。小样本循环网络的曲线会波动，因此验收同时看初末损失与训练集准确率。

In [ ]:
torch.manual_seed(SEED + 2)
model19 = SequenceClassifier(3, 14, 2, kind="lstm").to(DEVICE)
optimizer19 = torch.optim.Adam(model19.parameters(), lr=0.035)
losses19, raw_grad_norms = [], []

model19.train()
for step in range(180):
    optimizer19.zero_grad(set_to_none=True)
    logits, _ = model19(x_train, len_train)
    loss = F.cross_entropy(logits, y_train)
    loss.backward()
    grads = [p.grad for p in model19.parameters() if p.grad is not None]
    if step == 0:
        assert grads and all(torch.isfinite(g).all() for g in grads)
        assert sum(float(g.abs().sum()) for g in grads) > 0
    raw_norm = torch.nn.utils.clip_grad_norm_(model19.parameters(), max_norm=1.0)
    optimizer19.step()
    losses19.append(float(loss.detach()))
    raw_grad_norms.append(float(raw_norm))

model19.eval()
with torch.no_grad():
    train_logits, train_sequence = model19(x_train, len_train)
    train_pred = train_logits.argmax(-1)
    train_acc = (train_pred == y_train).float().mean().item()

assert losses19[-1] < losses19[0] * 0.25
assert train_acc >= 0.95
assert all(math.isfinite(v) for v in raw_grad_norms)
assert torch.count_nonzero(train_sequence[~valid_train]) == 0
print({"loss_first": losses19[0], "loss_last": losses19[-1], "controlled_train_acc": train_acc})

## 8. 失败反例：用 `x != 0` 猜 padding

真实特征完全可能为零，而 padding 也可能是非零占位。由内容反推 mask 会让相同有效序列因填充值不同而得到不同表示。可靠边界必须显式传 `lengths` 或上游给出的布尔 mask。

In [ ]:
probe = x_train[:3].clone()
probe_changed = probe.clone()
for b, length in enumerate(len_train[:3].tolist()):
    probe_changed[b, length:] = 999.0

with torch.no_grad():
    good_a, _ = model19(probe, len_train[:3])
    good_b, _ = model19(probe_changed, len_train[:3])
    # 错误做法：谎报所有位置有效，padding 就进入状态。
    wrong_lengths = torch.full_like(len_train[:3], probe.shape[1])
    bad_a, _ = model19(probe, wrong_lengths)
    bad_b, _ = model19(probe_changed, wrong_lengths)

assert torch.allclose(good_a, good_b, atol=1e-6)
assert not torch.allclose(bad_a, bad_b, atol=1e-3)
assert (bad_a - bad_b).abs().max().item() > 0.01
print("显式长度保持输出不变；错误长度会泄漏 padding。")

## 9. 状态化推理与分块等价性

流式推理可以把上一块的状态传入下一块，但必须明确会话边界、batch 重排规则和状态过期时间。下方验证：在无 padding 的同一序列上，“整段扫描”的末状态与“两块扫描”的末状态一致。生产系统若跨用户复用状态，会造成严重数据串线。

In [ ]:
stream_encoder = model19.encoder
stream_x = torch.randn(2, 6, 3, generator=torch.Generator().manual_seed(SEED + 3))
full_lengths = torch.tensor([6, 6])
with torch.no_grad():
    _, full_state = stream_encoder(stream_x, full_lengths)
    _, state_a = stream_encoder(stream_x[:, :2], torch.tensor([2, 2]))
    _, state_b = stream_encoder(stream_x[:, 2:], torch.tensor([4, 4]), state=state_a)

assert torch.allclose(full_state[0], state_b[0], atol=1e-6)
assert torch.allclose(full_state[1], state_b[1], atol=1e-6)
assert full_state[0].shape == (2, 14)

## 10. 保存合同：配置、权重指纹与可重复加载

仅保存 `state_dict` 不足以解释权重：还要保存 cell 类型、门顺序、维度、随机种子、PyTorch 版本和数据/标签协议。这里用内存缓冲区模拟制品，SHA-256 指纹可用于部署前校验字节是否被替换。

In [ ]:
manifest19 = {
    "artifact": "sequence_classifier",
    "schema_version": 1,
    "cell": "custom_lstm_ifgo",
    "input_size": 3,
    "hidden_size": 14,
    "classes": ["mean_non_positive", "mean_positive"],
    "seed": SEED,
    "torch_version": torch.__version__,
    "mask_contract": "lengths in [1,T], padding state frozen",
}
buffer19 = io.BytesIO()
torch.save({"manifest": manifest19, "state_dict": model19.state_dict()}, buffer19)
payload19 = buffer19.getvalue()
fingerprint19 = hashlib.sha256(payload19).hexdigest()

buffer19.seek(0)
loaded19 = torch.load(buffer19, map_location="cpu", weights_only=False)
clone19 = SequenceClassifier(3, 14, 2, kind="lstm")
clone19.load_state_dict(loaded19["state_dict"])
clone19.eval()
with torch.no_grad():
    clone_logits, _ = clone19(x_train, len_train)

assert loaded19["manifest"]["schema_version"] == 1
assert len(fingerprint19) == 64
assert torch.allclose(train_logits, clone_logits, atol=1e-7)
assert set(loaded19["state_dict"]) == set(model19.state_dict())
print({"sha256": fingerprint19[:16] + "…", "bytes": len(payload19)})

## 11. 生产观测与安全边界

上线至少记录：输入长度分布、超长截断率、空序列拒绝数、每批延迟、梯度范数、损失、类别分布漂移和模型指纹。状态化服务还要以租户与会话双键隔离状态，并在退出/超时后删除。

不要直接加载不可信的 pickle 权重；优先使用受控制品仓、哈希校验和 `weights_only=True` 能覆盖的格式。输入维度与长度应在进入模型前限流，否则循环长度本身就是拒绝服务面。

## 12. 何时替换教学实现

- 需要 GPU 吞吐、混合精度或长序列：换成官方融合循环层，并用本笔记的数值测试做回归。
- 需要跨框架导出：固定门顺序、GRU 变体、batch 维和初始状态语义。
- 需要真实泛化结论：按实体/用户/时间切分独立验证集与测试集，报告置信区间，而不是复用这里的过拟合准确率。
- 需要多层/双向网络：明确每层 dropout 的位置以及双向状态拼接顺序。

## 13. 原始资料与实现依据

- Elman, *Finding Structure in Time* (1990)：https://doi.org/10.1016/0364-0213(90)90002-E
- Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997)：https://doi.org/10.1162/neco.1997.9.8.1735
- Cho et al., *Learning Phrase Representations using RNN Encoder–Decoder* (2014)：https://arxiv.org/abs/1406.1078
- PyTorch `nn.Module` 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 梯度裁剪官方文档：https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html

公式来自论文；代码是为可解释性重新实现的教学版本。

## 14. 面试式复盘

1. LSTM 的 cell state 为何比纯 RNN 更容易保留梯度？关键是加法更新路径与门控。
2. mask 为什么既要冻结状态又要清零逐步输出？两者保护不同下游接口。
3. “训练准确率 100%”为什么不是好模型证据？这里只验证代码可学习，没有独立分布的泛化证明。
4. checkpoint 迁移最容易漏什么？门顺序、GRU 变体、标签映射、预处理和版本。
5. 流式状态为何属于安全问题？状态串会话会直接泄露另一用户的上下文。

In [ ]:
# 最终合同测试：覆盖拒绝非法长度、确定性推理与 cell 的梯度路径。
try:
    model19(torch.randn(2, 3, 3), torch.tensor([3, 0]))
    invalid_length_rejected = False
except ValueError:
    invalid_length_rejected = True

for kind in CELL_TYPES:
    gradient_probe = SequenceEncoder(3, 4, kind)
    probe_sequence, _ = gradient_probe(torch.randn(2, 3, 3), torch.tensor([3, 2]))
    probe_sequence.square().mean().backward()
    probe_grads = [p.grad for p in gradient_probe.parameters()]
    assert all(g is not None and torch.isfinite(g).all() for g in probe_grads)
    assert sum(float(g.abs().sum()) for g in probe_grads) > 0

with torch.no_grad():
    repeat_logits, _ = model19(x_train, len_train)

assert invalid_length_rejected
assert torch.allclose(train_logits, repeat_logits)
assert manifest19["cell"] == "custom_lstm_ifgo"
assert manifest19["mask_contract"].startswith("lengths")
assert losses19[-1] < 0.2
assert 0.0 <= train_acc <= 1.0
assert fingerprint19 == hashlib.sha256(payload19).hexdigest()
print("Notebook 19：全部合同测试通过。")